# BPClassifier Pipeline: Steps 1-7

This notebook is the thin, readable front end. Core logic lives in `src/project_notebook.py`; model hyperparameters live in `src/model_config.py`.

## 0. Setup

In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "paths.py").is_file():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Could not find the HW2 project root from this notebook.")

PROJECT_ROOT


PosixPath('/home/tian/code/baruch/NLP_HW/HW2')

## 1. Configuration

In [2]:
from src.model_config import ZooRunConfig
from src.project_notebook import print_environment_report

zoo_cfg = ZooRunConfig.for_8gb_gpu(seed=42, show_progress=True)
print_environment_report(zoo_cfg)


Environment:
{
  "project_root": "/home/tian/code/baruch/NLP_HW/HW2",
  "python": "3.10.20",
  "platform": "Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39",
  "cache_dir": "/home/tian/code/baruch/NLP_HW/HW2/cache",
  "gold_labeled_exists": true,
  "target_profile": "8 GB CUDA GPU defaults, CPU fallback if CUDA is unavailable",
  "torch": {
    "torch": "2.11.0+cu126",
    "cuda_available": true,
    "cuda_device": "NVIDIA GeForce RTX 4060 Laptop GPU",
    "cuda_memory_gb": 8.0
  },
  "model_config": {
    "sentence_transformers_model_id": "sentence-transformers/all-MiniLM-L6-v2",
    "embedding_batch_size": 256,
    "finbert_batch_size": 16,
    "finbert_eval_batch_size": 32,
    "setfit_batch_size": 128,
    "setfit_num_epochs": [
      1,
      2
    ],
    "setfit_num_iterations": 10,
    "skip_finbert": false,
    "skip_setfit": false
  }
}


## 2. Sentence Pool and Gold Labels

This uses cached artifacts when available. To force Haiku arbitration and rewrite `cache/gold_labeled.parquet`, pass `extra_argv=["--use-api-check"]` and set `ANTHROPIC_API_KEY`.

In [3]:
from src.project_notebook import run_gold_standard_pipeline_report

run_gold_standard_pipeline_report(seed=zoo_cfg.seed)


Gold pipeline: completed.
max_transcripts None max_pool_rows None gold_n_target 3000
loaded pool rows from cache 55485 -> /home/tian/code/baruch/NLP_HW/HW2/cache/sentence_pool.parquet
pool rows 55485
loaded gold sample from cache 3000 -> /home/tian/code/baruch/NLP_HW/HW2/cache/gold_sample.parquet
judge cache llama3.1:8b -> /home/tian/code/baruch/NLP_HW/HW2/cache/labels_llama3.1_8b.parquet
judge cache qwen3:8b-q4_K_M -> /home/tian/code/baruch/NLP_HW/HW2/cache/labels_qwen3_8b-q4_K_M.parquet
judge cache gemma2:9b -> /home/tian/code/baruch/NLP_HW/HW2/cache/labels_gemma2_9b.parquet
sample rows 3000
haiku_api_usage_preview
merge skipped (pass --use-api-check to call Anthropic and write gold_labeled.parquet)
gold_labeled exists rows 3000 cols ['api_error', 'api_used', 'discord', 'gold_final', 'j1_label', 'j1_parse_fallback', 'j1_raw', 'j2_label', 'j2_parse_fallback', 'j2_raw', 'j3_label', 'j3_parse_fallback', 'j3_raw', 'sentence_id', 'source_file', 'text', 'tie_break_label']
frac_needs_haiku_

## 3. Split and Handcrafted Features

In [4]:
from src.project_notebook import run_stratified_split_and_handcrafted_features_report

split_result = run_stratified_split_and_handcrafted_features_report(seed=zoo_cfg.seed)


Split and handcrafted features: completed.
Split core metadata:
{
  "rows_input": 3000,
  "rows_supervision": 3000,
  "seed": 42,
  "sentences_per_split": {
    "test": 543,
    "train": 1826,
    "val": 631
  },
  "transcripts_per_split": {
    "test": 26,
    "train": 79,
    "val": 26
  },
  "transcripts_total": 131
}
            sentence_id     source_file  gold_final split
AMD_Q1-2024.txt#0000154 AMD_Q1-2024.txt substantive train
AMD_Q1-2024.txt#0000052 AMD_Q1-2024.txt substantive train
AMD_Q1-2024.txt#0000346 AMD_Q1-2024.txt substantive train
AMD_Q1-2024.txt#0000038 AMD_Q1-2024.txt substantive train
AMD_Q1-2024.txt#0000174 AMD_Q1-2024.txt boilerplate train
 h01  h02  h03  h04  h05  h06  h07  h08
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
feature_columns: 30


## 4. Classifier Zoo, Ensembles, and Winner

The default config runs the full family set. For a quick local smoke check, use `ZooRunConfig.quick_non_transformer(seed=42)` instead.

In [5]:
from src.project_notebook import run_classifier_zoo_report

zoo_result = run_classifier_zoo_report(cfg=zoo_cfg)


Classifier zoo config:
Key hyperparameters:
{
  "recall_floor": 0.96,
  "embedding_batch_size": 256,
  "finbert_batch_size": 16,
  "finbert_eval_batch_size": 32,
  "setfit_batch_size": 128,
  "setfit_num_epochs": [
    1,
    2
  ],
  "setfit_num_iterations": 10,
  "skip_finbert": false,
  "skip_setfit": false
}
[zoo] start (OOF + test)


/home/tian/opt/miniconda/envs/nlp/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[zoo] sentence embeddings: all supervised rows …
[zoo] B/C/D OOF models …


tokenize: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1976/1976 [00:00<00:00, 55433.68 examples/s]

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 31678.32it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |                                                                                       
-----------------------------+------------+---------------------------------------------------------------------------------------
bert.embeddings.position_ids | UNEXPECTED |                                                                                       
classifier.bias          

Step,Training Loss


...

Step,Training Loss
1,0.369856


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3780.97it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Map: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2457/2457 [00:00<00:00, 69324.77 examples/s]
***** Running training *****
  Num unique pairs = 49140
  Batch size = 128
  Num epochs = 1


Step,Training Loss
1,0.330424


Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 3306.37it/s]


[zoo] done.
          family_id  test_macro_f1  test_accuracy  test_precision_boilerplate  test_recall_boilerplate  test_f1_boilerplate  test_precision_substantive  test_recall_substantive  test_f1_substantive  test_confusion_matrix  eligible_oof  threshold  threshold_std_across_folds  train_seconds  approx_infer_sents_per_sec notes
          E_finbert       0.866598       0.889503                    0.871622                 0.758824             0.811321                    0.896203                 0.949062             0.921875 [[129, 41], [19, 354]]          True      0.148                    0.071334      88.816718                5.512689e+02      
           F_setfit       0.841078       0.867403                    0.822368                 0.735294             0.776398                    0.884910                 0.927614             0.905759 [[125, 45], [27, 346]]          True      0.028                    0.019467     886.900619                3.318591e+03      
    G_ensemble_mean